# Lecture 2 · Notebook 3 — Word2vec from scratch, and how to evaluate word vectors
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture02/lecture02_3_word2vec_from_scratch.ipynb)

**Companion to [Chapters 8–10 of the video lecture](https://nlp-course-cuhksz.github.io/lecture02.html#p8-s156)** (slides 149–194). Word2vec learns word vectors by predicting
which words appear near each other. In this notebook you will:

1. turn raw text into skip-gram **(center, context) training pairs** (slides 156–157),
2. measure why the **full softmax** is expensive and how **negative sampling** avoids it (slides 159, 164–169, 194),
3. train **skip-gram with negative sampling (SGNS)** in PyTorch on 1.7 M words of Wikipedia,
4. explore neighbors and **analogies** with your vectors and with pretrained **GloVe** (slides 150–151, 172–173),
5. **evaluate** both with WordSim-353 (Spearman) and the Google analogy test (slides 177–180), and
6. probe the vectors for **gender bias** (slide 154).

Runs on the free CPU runtime in about 10 minutes, most of it training (downloads: 10 MB of `text8`, the 171 MB GloVe file, two small test sets).

In [ ]:
import os, time, math, collections, requests
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import spearmanr
torch.manual_seed(0); np.random.seed(0)

## 1 · Text → training pairs
We use the first 10 MB of `text8` (cleaned English Wikipedia) and keep words that occur at least 5 times. Skip-gram slides a window over the
text: each **center** word is paired with every **context** word up to `m` positions away (slide 156).

In [ ]:
URL = "https://huggingface.co/datasets/ardMLX/text8/resolve/main/text8"     # a copy of Matt Mahoney's text8
if not os.path.exists("text8_10MB.txt"):
    r = requests.get(URL, headers={"Range": "bytes=0-9999999"}, timeout=300)   # only the first 10 MB
    open("text8_10MB.txt", "w").write(r.text)
words = open("text8_10MB.txt").read().split()[:-1]
cnt = collections.Counter(words)
vocab = [w for w, c in cnt.most_common() if c >= 5]
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
ids = np.array([w2i[w] for w in words if w in w2i], dtype=np.int64)
freq = np.array([cnt[w] for w in vocab], dtype=np.float64)
print(f"{len(words):,} tokens, vocabulary |V| = {V:,} words (count >= 5), {len(ids):,} tokens kept")

def pairs_for(sentence, m=2):
    toks = sentence.split()
    return [(toks[t], toks[t + j]) for t in range(len(toks)) for j in range(-m, m + 1) if j != 0 and 0 <= t + j < len(toks)]

example = "problems turning into banking crises as"          # the example of slide 157
print([p for p in pairs_for(example) if p[0] in ("into", "banking")])

Two tricks from the word2vec paper make training faster and better: **subsampling** randomly drops very frequent words such as *the*
(they carry little information as context), and a **random window size** between 1 and `m` gives nearby words more weight.

In [ ]:
WINDOW = 5
f = freq / freq.sum()
p_keep = np.minimum(1.0, np.sqrt(1e-4 / f) + 1e-4 / f)       # word2vec's subsampling rule, threshold t = 1e-4
for w in ["the", "of", "king", "coffee"]:
    print(f"P(keep {w!r:9}) = {p_keep[w2i[w]]:.2f}")

def make_pairs(ids):
    s = ids[np.random.rand(len(ids)) < p_keep[ids]]
    win = np.random.randint(1, WINDOW + 1, size=len(s))      # each center gets its own window size
    centers, contexts = [], []
    for off in range(1, WINDOW + 1):
        m = win[:-off] >= off; centers.append(s[:-off][m]); contexts.append(s[off:][m])    # context to the right
        m = win[off:] >= off;  centers.append(s[off:][m]);  contexts.append(s[:-off][m])   # context to the left
    c, o = np.concatenate(centers), np.concatenate(contexts)
    perm = np.random.permutation(len(c))
    return torch.from_numpy(c[perm]), torch.from_numpy(o[perm])

c_all, o_all = make_pairs(ids)
print(f"{len(c_all):,} (center, context) pairs for one epoch, e.g.", [(vocab[a], vocab[b]) for a, b in zip(c_all[:5].tolist(), o_all[:5].tolist())])

## 2 · The model: two embedding tables
Every word has a **center vector** $u_w$ and a **context vector** $v_w$ (slides 159 and 163), so the model has $2|V|d$ parameters (slide 162).
With full softmax, $P(o \mid c) = \exp(u_c^\top v_o) / \sum_{w \in V} \exp(u_c^\top v_w)$ — the denominator touches **every** context
vector. Let's time one training step of each version on the same batch.

In [ ]:
D = 100
print(f"parameters: 2 x |V| x d = 2 x {V:,} x {D} = {2 * V * D:,}")

U_t, V_t = nn.Embedding(V, D, sparse=True), nn.Embedding(V, D, sparse=True)
batch_c, batch_o = c_all[:1024], o_all[:1024]
K = 5
noise = freq ** 0.75; noise /= noise.sum()                     # unigram^(3/4) noise distribution (slide 167)
table = torch.from_numpy(np.repeat(np.arange(V), np.round(noise * 1e7).astype(np.int64)))   # sampling table, as in word2vec

def full_softmax_step():
    logits = U_t(batch_c) @ V_t.weight.T                      # [B, |V|]: a score for every word in the vocabulary
                                                              # (the gradient reaches all |V| context vectors)
    loss = F.cross_entropy(logits, batch_o)
    loss.backward()

def sgns_step():
    neg = table[torch.randint(0, len(table), (len(batch_c), K))]   # K noise words per pair
    u = U_t(batch_c)
    pos = (u * V_t(batch_o)).sum(-1)                          # [B]
    negs = torch.bmm(V_t(neg), u.unsqueeze(-1)).squeeze(-1)   # [B, K]
    loss = -(F.logsigmoid(pos) + F.logsigmoid(-negs).sum(-1)).mean()
    loss.backward()

for name, fn in [("full softmax", full_softmax_step), ("negative sampling (K=5)", sgns_step)]:
    t0 = time.time()
    for _ in range(10): fn()
    print(f"{name:24} {(time.time() - t0) / 10 * 1000:7.1f} ms per batch of 1,024 pairs")
print(f"\nmultiply-adds per pair for the scores: full softmax |V|*d = {V * D:,}  vs  SGNS (K+1)*d = {(K + 1) * D:,}")
print(f"rows of the context table touched per pair: full softmax {V:,}  vs  SGNS K+1 = {K + 1} (plus 1 center row, slide 169)")

The arithmetic differs by a factor of $|V|/(K+1)$, about 3,000 here. The wall-clock gap on a CPU is smaller, because small operations
have fixed overheads, but the full-softmax cost grows linearly with the vocabulary (slide 194) while the SGNS cost does not.

The 3/4 power in the noise distribution (slide 167) flattens the unigram distribution, so rare words are drawn as negatives a bit more often:

In [ ]:
p_uni = freq / freq.sum()
for w in ["the", "king", "coffee", vocab[-1]]:
    i = w2i[w]
    print(f"{w:10} unigram {p_uni[i]:.2e}   unigram^0.75 {noise[i]:.2e}   ratio x{noise[i] / p_uni[i]:.1f}")

## 3 · Train skip-gram with negative sampling
For each observed pair the loss is $-\log\sigma(u_c^\top v_o) - \sum_{k=1}^{K}\log\sigma(-u_c^\top v_{n_k})$ (slides 166–167):
push the true pair's score up and the sampled noise pairs' scores down. The embeddings are *sparse*: each step updates only the rows used
in the batch (the K + 2 vectors per pair of slide 169), here with the `SparseAdam` optimizer. Each epoch takes one to two minutes on a
Colab CPU; set `EPOCHS = 1` if you are in a hurry.

In [ ]:
EPOCHS, B, LR = 3, 4096, 0.005
U = nn.Embedding(V, D, sparse=True)                 # center vectors u
Vc = nn.Embedding(V, D, sparse=True)                # context vectors v
nn.init.uniform_(U.weight, -0.5 / D, 0.5 / D); nn.init.zeros_(Vc.weight)
opt = torch.optim.SparseAdam(list(U.parameters()) + list(Vc.parameters()), lr=LR)

t0 = time.time()
for ep in range(EPOCHS):
    c_all, o_all = make_pairs(ids)                  # fresh subsampling and windows every epoch
    n_steps = (len(c_all) + B - 1) // B
    running = 0.0
    for step in range(n_steps):
        cb, ob = c_all[step * B:(step + 1) * B], o_all[step * B:(step + 1) * B]
        neg = table[torch.randint(0, len(table), (len(cb), K))]
        u = U(cb)
        pos = (u * Vc(ob)).sum(-1)
        negs = torch.bmm(Vc(neg), u.unsqueeze(-1)).squeeze(-1)
        loss = -(F.logsigmoid(pos) + F.logsigmoid(-negs).sum(-1))
        loss = loss.mean()
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
        if (step + 1) % 400 == 0 or step == n_steps - 1:
            print(f"epoch {ep + 1}  step {step + 1:4d}/{n_steps}  loss {running / (step % 400 + 1):.3f}  ({time.time() - t0:.0f}s)")
            running = 0.0

E_sg = U.weight.detach().numpy()
E_sg = E_sg / np.linalg.norm(E_sg, axis=1, keepdims=True)

## 4 · Explore: neighbors and analogies
Cosine similarity finds neighbors (slide 150). An analogy *a : b :: c : ?* is answered by the word closest to $b - a + c$, excluding the
three query words (slides 151 and 179).

In [ ]:
def neighbors(E, w2i, vocab, word, k=6):
    s = E @ E[w2i[word]]
    s[w2i[word]] = -np.inf
    return [vocab[i] for i in np.argsort(-s)[:k]]

def analogy(E, w2i, vocab, a, b, c, k=3):
    q = E[w2i[b]] - E[w2i[a]] + E[w2i[c]]
    s = E @ (q / np.linalg.norm(q))
    for x in (a, b, c): s[w2i[x]] = -np.inf
    return [vocab[i] for i in np.argsort(-s)[:k]]

for w in ["sweden", "king", "computer", "red", "three", "einstein"]:
    print(f"{w:9} -> {', '.join(neighbors(E_sg, w2i, vocab, w))}")
print()
for a, b, c in [("man", "king", "woman"), ("france", "paris", "germany"), ("big", "bigger", "small")]:
    print(f"{a} : {b} :: {c} : ?  ->", analogy(E_sg, w2i, vocab, a, b, c))

Neighbors already look sensible after a few minutes on 1.7 M words; analogies are much harder and need far more text. Pretrained vectors
were trained on billions of words (slide 172). Let's load **GloVe** (6 B tokens of Wikipedia + Gigaword, 50 dimensions) and keep its
100,000 most frequent words.

In [ ]:
from huggingface_hub import hf_hub_download
path = hf_hub_download("antokun/glove.6B.50d", "glove.6B.50d.txt", repo_type="dataset")   # the original Stanford file, 171 MB
g_vocab, g_vecs = [], []
with open(path, encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if i == 100_000: break                      # the file is sorted by frequency
        parts = line.rstrip().split(" ")
        g_vocab.append(parts[0]); g_vecs.append(np.asarray(parts[1:], dtype=np.float32))
E_gl = np.stack(g_vecs); E_gl /= np.linalg.norm(E_gl, axis=1, keepdims=True)
g_w2i = {w: i for i, w in enumerate(g_vocab)}

for w in ["sweden", "king", "computer", "frog"]:
    print(f"{w:9} -> {', '.join(neighbors(E_gl, g_w2i, g_vocab, w))}")
print()
for a, b, c in [("man", "king", "woman"), ("chicago", "illinois", "philadelphia"), ("bad", "worst", "cool"), ("france", "paris", "japan")]:
    print(f"{a} : {b} :: {c} : ?  ->", analogy(E_gl, g_w2i, g_vocab, a, b, c))

GloVe's neighbors are clean and it solves *king → queen* and *Paris → Tokyo*, but these 50-dimensional vectors — the smallest GloVe
release — still miss the two examples of slide 179 (*Pennsylvania*, *coolest*). Vector arithmetic captures some relations, not all
(slide 151: "an approximate pattern, not a guaranteed rule").

## 5 · Intrinsic evaluation
**Word similarity** (slide 177): WordSim-353 has 353 word pairs rated by humans from 0 to 10. We compute the **Spearman correlation**
between human ratings and cosine similarities — only the *ranking* has to agree. **Word analogy** (slides 179–180): the Google test set has
19,544 questions in semantic (capital–country, family…) and syntactic (plural, past tense…) groups. We only score questions whose four
words are all in the vocabulary, and report that coverage too.

In [ ]:
def fetch(url, path):
    if not os.path.exists(path):
        open(path, "w", encoding="utf-8").write(requests.get(url, timeout=120).text)
    return open(path, encoding="utf-8").read().splitlines()

ws = [l.split("\t") for l in fetch("https://raw.githubusercontent.com/piskvorky/gensim/develop/gensim/test/test_data/wordsim353.tsv",
                                   "wordsim353.tsv") if l and not l.startswith("#")]
ws = [(a.lower(), b.lower(), float(s)) for a, b, s in ws]
qs = fetch("https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt", "questions-words.txt")

def wordsim(E, w2i):
    ok = [(a, b, s) for a, b, s in ws if a in w2i and b in w2i]
    rho = spearmanr([s for *_, s in ok], [float(E[w2i[a]] @ E[w2i[b]]) for a, b, _ in ok]).correlation
    return rho, len(ok)

def analogy_eval(E, w2i, max_per_section=300):
    res, section = collections.defaultdict(lambda: [0, 0, 0]), None
    for line in qs:
        if line.startswith(":"):
            section = "semantic" if not line.startswith(": gram") else "syntactic"; seen = 0; continue
        a, b, c, d = line.lower().split()
        res[section][2] += 1
        if not all(x in w2i for x in (a, b, c, d)) or seen >= max_per_section: continue
        seen += 1
        q = E[w2i[b]] - E[w2i[a]] + E[w2i[c]]
        s = E @ q
        for x in (a, b, c): s[w2i[x]] = -np.inf
        res[section][0] += int(np.argmax(s) == w2i[d]); res[section][1] += 1
    return {k: (v[0] / max(v[1], 1), v[1], v[2]) for k, v in res.items()}

for name, E, idx in [("our SGNS (1.7 M words, d=100)", E_sg, w2i), ("GloVe (6 B tokens, d=50)", E_gl, g_w2i)]:
    rho, n = wordsim(E, idx)
    print(f"{name}\n   WordSim-353 Spearman = {rho:.3f}  on {n}/{len(ws)} pairs")
    for sec, (acc, n_ok, n_all) in analogy_eval(E, idx).items():
        print(f"   analogy {sec:9} accuracy = {acc:.1%}  on {n_ok:,} scored questions ({n_all:,} in the set)")

Two lessons from these numbers. First, **different tests can disagree** (slides 178 and 180): our small model can score as well as
50-dimensional GloVe on WordSim-353 (and it is only scored on the pairs it knows), yet it solves far fewer analogies and cannot even attempt
most of them, because the words are missing from its vocabulary. Second, **data and dimension matter**: on slide 178, GloVe and skip-gram
trained on 6 billion tokens with 300 dimensions reach about 0.63–0.66 on WS353. And good intrinsic scores still do not guarantee good
results in a real task (slide 175).

## 6 · Bias in embeddings
Embeddings learn the associations in their training text, including stereotypes (slide 154). One simple probe projects occupation words
onto the *he − she* direction: positive means closer to *he*, negative closer to *she*.

In [ ]:
g = E_gl[g_w2i["he"]] - E_gl[g_w2i["she"]]
g /= np.linalg.norm(g)
jobs = ["nurse", "receptionist", "librarian", "teacher", "secretary", "doctor", "scientist", "engineer", "programmer", "carpenter", "captain", "pilot"]
for w, s in sorted(((w, float(E_gl[g_w2i[w]] @ g)) for w in jobs), key=lambda x: x[1]):
    bar = "#" * int(abs(s) * 40)
    print(f"{w:13} {s:+.3f}  {bar:>12}|" if s < 0 else f"{w:13} {s:+.3f}  {'':>12}|{bar}")

These numbers describe **co-occurrence patterns in news and Wikipedia text**, not facts about people. Some surprises have a linguistic
explanation: *secretary* may lean towards *he* because news text is full of *Secretary of State* and *Defense Secretary* — one static
vector has to mix all senses of a word (slide 173). If such vectors feed a hiring or
recommendation system, the associations can influence real decisions — a reason to inspect embeddings, not just their average accuracy.

Finally, a reminder of slide 176: a sentence vector built by **adding** word vectors cannot see word order.

In [ ]:
sent_vec = lambda s: sum(E_gl[g_w2i[w]] for w in s.split())
a, b = sent_vec("the dog chased the cat"), sent_vec("the cat chased the dog")
print("identical sentence vectors?", np.allclose(a, b))

## Exercises
1. Train with `WINDOW = 2` and with `WINDOW = 10`. How do the neighbors of `running`, `red` and `einstein` change?
2. Set `K = 1` and `K = 15` negative samples. What happens to training time and to the WordSim-353 score?
3. Replace `noise` by the plain unigram distribution (no 3/4 power) for the sampling table. Does the analogy accuracy change?
4. Average the two tables, `U.weight + Vc.weight`, instead of using `U` alone (slide 163). Is the result better on WordSim-353?
5. The appendix (slides 183–194) derives $\partial J / \partial u_c = \sum_w P(w \mid c)\, v_w - v_o$ for full softmax. Check it with
   `torch.autograd` on a tiny random example.

**Back to:** [Notebook 1 — from bytes to tokens](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture02/lecture02_1_from_bytes_to_tokens.ipynb) · [the video lecture](https://nlp-course-cuhksz.github.io/lecture02.html)